use pyspark sql

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [ ]:
spark = SparkSession.builder.master("local[*]").appName("spark_sql").getOrCreate()

In [ ]:
df = (spark.read
      .option("header", "true")
      .option("inferSchema", "true")
      .option("delimiter", ";")
      .csv("data/01.csv")
      )

In [ ]:
df.printSchema()

In [ ]:
# Rename column names, removing spaces
# Each column must be a unique value, not a list
# Use * to unpack the list and pass clean values to toDF()
df = df.toDF(*[c.strip().lower().replace("-", "").replace("  ","_").replace(" ", "_") for c in df.columns])
df.show()

In [ ]:
#to simulate a sql table
df.createOrReplaceTempView("fuels")

In [ ]:
spark.sql("""desc fuels""").show()

In [ ]:
spark.sql("""select * from fuels""").show()

In [ ]:
spark.sql("""select revenda, estado_sigla from fuels group by estado_sigla, revenda order by estado_sigla asc""").show(100)

In [ ]:
spark.sql("""select * from fuels where valor_de_compra is not null""").show()

In [ ]:
spark.sql("""
    SELECT 
        estado_sigla, 
        produto, 
        (
            MAX(CAST(REPLACE(valor_de_venda, ',', '.') AS DECIMAL(3,2))) - 
            MIN(CAST(REPLACE(valor_de_venda, ',', '.') AS DECIMAL(3,2)))
        ) AS diferenca
    FROM fuels 
    GROUP BY estado_sigla, produto
    ORDER BY 1,3
""").show()